In [ ]:
#| default_exp energy

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import logging, os, time, warnings
from dataclasses import dataclass, asdict
from typing import Sequence

import torch

from fasterbench.core import _device_ctx, _sync, _run_on_devices

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    # Suppress codecarbon's verbose logging BEFORE any tracker is constructed,
    # because codecarbon logs messages in __init__ before applying log_level.
    logging.getLogger("codecarbon").setLevel(logging.CRITICAL)
except ImportError:
    EmissionsTracker = OfflineEmissionsTracker = None

In [ ]:
#| export
@dataclass(slots=True)
class EnergyMetrics:
    """Energy consumption and carbon footprint metrics."""
    mean_watts: float   # average power during measurement
    energy_wh: float    # Wh per inference
    co2_eq_g: float     # g CO₂-eq per inference

    def as_dict(self) -> dict[str, float]:
        return asdict(self)


#| export
def _nan_energy_metrics(device: str) -> EnergyMetrics:  # device string (unused, for consistent signature)
    """Create EnergyMetrics with NaN values for failed benchmarks."""
    nan = float("nan")
    return EnergyMetrics(nan, nan, nan)


#| export
def _clear_stale_codecarbon_lock() -> None:
    """Remove stale codecarbon lock file if the owning process no longer exists."""
    import tempfile
    lock_path = os.path.join(tempfile.gettempdir(), ".codecarbon.lock")
    if not os.path.exists(lock_path):
        return
    try:
        # Read the PID from the lock file (codecarbon writes its PID there)
        with open(lock_path) as f:
            content = f.read().strip()
        if content:
            pid = int(content)
            os.kill(pid, 0)  # Check if process exists (signal 0 = no-op)
            # Process exists — lock is valid, don't remove
            return
    except (ValueError, ProcessLookupError, PermissionError, OSError):
        pass  # PID invalid or process dead — lock is stale
    try:
        os.remove(lock_path)
    except OSError:
        pass


#| export
@torch.inference_mode()
def compute_energy(
    model: torch.nn.Module,                  # model to benchmark
    sample: torch.Tensor,                    # input tensor (with batch dimension)
    *,
    device: str | torch.device = "cpu",      # device to run on
    warmup: int = 20,                        # warmup iterations
    steps: int = 100,                        # measurement iterations
    offline: bool = True,                    # use offline emissions tracker
    country_iso: str | None = None,          # country ISO code for carbon intensity
    measure_secs: int = 1,                   # power sampling interval
) -> EnergyMetrics:
    """Measure power consumption and carbon footprint using codecarbon."""
    if EmissionsTracker is None:
        warnings.warn("codecarbon not installed – returning NaNs")
        return _nan_energy_metrics(str(device))

    _clear_stale_codecarbon_lock()

    Tracker = OfflineEmissionsTracker if offline else EmissionsTracker
    tracker = Tracker(
        project_name="fasterbench",
        country_iso_code=(country_iso or os.getenv("NNBENCH_ISO", "USA")),
        measure_power_secs=measure_secs,
        save_to_file=False,
        log_level="critical",
    )

    with _device_ctx(device) as dev:
        model = model.eval().to(dev)
        sample = sample.to(dev, non_blocking=True)

        for _ in range(warmup):
            model(sample)
        _sync(dev)

        tracker.start()
        try:
            t0 = time.perf_counter()
            for _ in range(steps):
                model(sample)
            _sync(dev)
        finally:
            tracker.stop()
        dur_s = time.perf_counter() - t0

    # codecarbon silently fails if another instance is running,
    # leaving final_emissions_data as None
    if tracker.final_emissions_data is None:
        warnings.warn("codecarbon tracker did not collect data (another instance may be running)")
        return _nan_energy_metrics(str(device))

    ene_kwh = tracker.final_emissions_data.energy_consumed
    co2_kg = tracker.final_emissions
    mean_w = (ene_kwh * 3600_000) / dur_s

    return EnergyMetrics(
        mean_watts=mean_w,
        energy_wh=(ene_kwh * 1_000) / steps,
        co2_eq_g=(co2_kg * 1_000) / steps,
    )


#| export
def compute_energy_multi(
    model: torch.nn.Module,                                # model to benchmark
    sample: torch.Tensor,                                  # input tensor (with batch dimension)
    *,
    devices: Sequence[str | torch.device] | None = None,   # devices to benchmark (default: cpu + cuda)
    **kwargs,
) -> dict[str, EnergyMetrics]:
    """Measure energy on multiple devices."""
    return _run_on_devices(
        compute_energy, model, sample, devices,
        nan_factory=_nan_energy_metrics,
        metric_name="Energy",
        **kwargs
    )

In [ ]:
show_doc(EnergyMetrics)

In [ ]:
show_doc(compute_energy)

In [ ]:
show_doc(compute_energy_multi)

---

## See Also

- [Memory](memory.html) — Memory measurement
- [Benchmark](../analysis/benchmark.html) — Unified API